# Round 2 — result figures

Reads the four round-2 runs (`train_student.py`) and their joint evaluation (`evaluate_alignment.py`). Files are only read, never modified.

**Models**
- **target**: Tiannan's trained DyRCNNx4 (frozen). Gives the pseudo-labels (its final prediction) and the layer representations.
- **control**: student trained with CE vs the pseudo-labels only. Shared by every aligned configuration.
- **aligned_IT_w1**: CE + 1.0 · log(1 − CKA_IT). Round-1 setting, re-run without data augmentation.
- **aligned_IT_w0p1**: CE + 0.1 · log(1 − CKA_IT). Does a smaller CKA weight let the CE go further down?
- **aligned_V1V4IT_w0p1**: CE + 0.1 · Σ over V1, V4, IT of log(1 − CKA). Aligning three layers at once.

All students: same architecture as the target, same initial weights, same data in the same order; alignment at the last timestep. Students are evaluated at their **last** checkpoint (epoch 199).

**Two data sources**
- *Training curves* (x-axis = epoch): `metrics/metrics.csv` of each run, measured on **val** (20% of the official train folder) during training.
- *Test results* (x-axis = timestep): the evaluation, on the **test** set (official val folder, 3925 images, never seen by the target).

Timesteps are numbered 0…19. CKA in this notebook is always the **biased** linear CKA (the same formula as the training loss); the unbiased values are in `cka_over_time.csv` if needed.

**Sections**: 0 target · A aligned_IT_w1 · B aligned_IT_w0p1 · C aligned_V1V4IT_w0p1 · D comparison across configurations.

Only edit the first code cell, then run all cells top to bottom.

In [ ]:
RUNS_ROOT = "/scratch/wz2723/dynvision/runs/round2"
EVAL_NAME = None   # subfolder of RUNS_ROOT/eval/; None = the most recent one

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd

eval_root = Path(RUNS_ROOT) / "eval"
eval_dir = (max((p for p in eval_root.iterdir() if p.is_dir()), key=lambda p: p.stat().st_mtime)
            if EVAL_NAME is None else eval_root / EVAL_NAME)

info = json.loads((eval_dir / "eval_info.json").read_text())
summary = json.loads((eval_dir / "summary.json").read_text())
over_time = pd.read_csv(eval_dir / "metrics_over_time.csv")
cka = pd.read_csv(eval_dir / "cka_over_time.csv")

RUNS = {r["name"]: r for r in info["runs"]}
CONTROL = next(n for n, r in RUNS.items() if r["is_control"])
ALIGNED = [n for n, r in RUNS.items() if not r["is_control"]]
hist = {n: pd.read_csv(Path(r["dir"]) / "metrics" / "metrics.csv") for n, r in RUNS.items()}

LAYERS = info["layers"]
T = int(over_time["timestep"].max()) + 1
LAST_T = T - 1
CE_FIRST_T = info["ce_first_timestep"]
KEY = {layer: f"{layer}_t{LAST_T:02d}" for layer in LAYERS}   # metrics.csv column suffix

# ---- style ----------------------------------------------------------------
COLORS = {"target": "#52514e", "aligned": "#2a78d6", "control": "#eb6834"}
RUN_COLORS = {CONTROL: "#eb6834"}
for name, color in zip(ALIGNED, ["#1f4e9c", "#6aa6e8", "#8e5bd0", "#2e9e8f", "#c2457a"]):
    RUN_COLORS[name] = color
TERM_COLORS = {"CE": "#52514e", "V1": "#3f9b4f", "V4": "#d39a1e", "IT": "#8e5bd0"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8984",
    "lines.linewidth": 1.5, "lines.markersize": 3.5, "legend.frameon": False,
    "axes.titlesize": 11, "axes.labelsize": 10, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.formatter.useoffset": False,   # no "1e-5 + 0.9995" offsets on near-constant curves
})

def key(label, color, ls="-", marker=None):
    return Line2D([], [], color=color, linestyle=ls, marker=marker, label=label)

def shared_legend(fig, handles, ncol=None, top=0.90):
    # One legend above all panels, so it never covers data.
    fig.legend(handles=handles, loc="upper center", ncol=ncol or len(handles), bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout(rect=(0, 0, 1, top))

def mark_time_axes(ax):
    # Grey = timesteps without CE supervision in training; dotted = the aligned timestep.
    ax.axvspan(-0.5, CE_FIRST_T - 0.5, color="#f0efeb", zorder=0)
    ax.axvline(LAST_T, color="#8a8984", linestyle=":", linewidth=1.2)
    ax.set_xlabel("timestep")
    ax.set_xticks(range(0, T, 2))

def epoch_axis(ax):
    ax.set_xlabel("epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

def test_metric(model, col):
    d = over_time[over_time["model"] == model].sort_values("timestep")
    return d["timestep"], d[col]

def test_cka(model, layer, partner="target"):
    d = cka[(cka["model_a"] == model) & (cka["model_b"] == partner) & (cka["layer"] == layer)]
    d = d.sort_values("timestep")
    return d["timestep"], d["cka_biased"]

def aligned_layers(name):
    return [l for l in LAYERS if KEY[l] in RUNS[name]["loss_layers"]]

print("eval:   ", eval_dir)
print("runs:   ", list(RUNS), "| control:", CONTROL)
print("epochs: ", {n: len(h) for n, h in hist.items()}, "| checkpoint:", info["args"]["checkpoint"],
      {n: r["epoch"] for n, r in RUNS.items()})
print(f"test images: {summary['n_test_images']} | layers: {LAYERS} | CE from timestep {CE_FIRST_T}")

In [ ]:
# ===== figure functions (used by sections A, B, C) ===========================

def fig_training(name):
    # X.1 — CE, val batch CKA of the aligned layers, gradient decomposition; x = epoch.
    al, ctl = hist[name], hist[CONTROL]
    layers = aligned_layers(name)

    # Row 1: CE vs pseudo-label
    fig, ax = plt.subplots(figsize=(12, 3.4))
    for n, df, color in ((name, al, COLORS["aligned"]), (CONTROL, ctl, COLORS["control"])):
        ax.plot(df["epoch"], df["val_ce_loss"], "-", color=color)
        ax.plot(df["epoch"], df["train_ce_loss"], "--", color=color, alpha=0.7)
    ax.set_title("CE vs pseudo-label (lower = better)")
    epoch_axis(ax)
    shared_legend(fig, [key(name, COLORS["aligned"]), key("control", COLORS["control"]),
                        key("val", "#52514e", "-"), key("train", "#52514e", "--")], top=0.86)
    plt.show()

    # Row 2: val batch CKA vs target, one panel per aligned layer
    fig, axes = plt.subplots(1, len(layers), figsize=(max(7.0, 4.2 * len(layers) + 0.6), 3.4), squeeze=False)
    for ax, layer in zip(axes[0], layers):
        for df, color in ((al, COLORS["aligned"]), (ctl, COLORS["control"])):
            ax.plot(df["epoch"], df[f"val_cka_{KEY[layer]}"], "-", color=color)
        ax.set_title(f"val batch CKA vs target — {layer} (t={LAST_T})")
        epoch_axis(ax)
    shared_legend(fig, [key(f"{name} vs target", COLORS["aligned"]),
                        key("control vs target", COLORS["control"])], top=0.84)
    plt.show()

    # Row 3: gradient decomposition (fixed diagnostic batch, once per epoch)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.6))
    ax1.plot(al["epoch"], al["diag_grad_ce"], "-", color=TERM_COLORS["CE"])
    ax1.plot(ctl["epoch"], ctl["diag_grad_ce"], "--", color=COLORS["control"])
    handles = [key(f"CE term ({name})", TERM_COLORS["CE"]), key("CE term (control)", COLORS["control"], "--")]
    for layer in layers:
        col = f"diag_grad_cka_{KEY[layer]}"
        w = RUNS[name]["loss_weights"][KEY[layer]]
        ax1.plot(al["epoch"], al[col], "-", color=TERM_COLORS[layer])
        handles.append(key(f"CKA term {layer} (w={w:g})", TERM_COLORS[layer]))
    ax1.set_yscale("log")
    ax1.set_title("gradient norm of each loss term (with its weight)")
    epoch_axis(ax1)
    ax2.plot(al["epoch"], al["diag_ratio_cka_to_ce"], "-", color=COLORS["aligned"])
    ax2.axhline(1.0, color="#b5b4af", linestyle="--", linewidth=1)
    ax2.set_yscale("log")
    ax2.set_title("|sum of CKA-term gradients| / |CE gradient|   (>1: CKA dominates)")
    epoch_axis(ax2)
    shared_legend(fig, handles, ncol=min(len(handles), 5), top=0.80)
    plt.show()


def fig_behavior(name):
    # X.2 — test set, x = timestep: GT accuracy, CE vs pseudo-label, agreement with target.
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    ax = axes[0]
    for m, color, ls, z in ((name, COLORS["aligned"], "-", 2), (CONTROL, COLORS["control"], "-", 2),
                            ("target", COLORS["target"], "--", 3)):
        x, y = test_metric(m, "gt_accuracy")
        ax.plot(x, y, ls, color=color, marker="o", zorder=z)
    ax.axhline(0.1, color="#b5b4af", linestyle="--", linewidth=1)
    ax.set_ylim(0, 1)
    ax.set_title("accuracy vs ground truth")
    mark_time_axes(ax)

    ax = axes[1]
    for m, color in ((name, COLORS["aligned"]), (CONTROL, COLORS["control"])):
        x, y = test_metric(m, "ce_pseudo")
        ax.plot(x, y, "-", color=color, marker="o")
    ax.set_yscale("log")
    ax.set_title("CE vs pseudo-label (log scale)")
    mark_time_axes(ax)

    ax = axes[2]
    for m, color, ls in ((name, COLORS["aligned"], "-"), (CONTROL, COLORS["control"], "-"),
                         ("target", COLORS["target"], "--")):
        x, y = test_metric(m, "agreement")
        ax.plot(x, y, ls, color=color, marker="o")
    ax.set_ylim(0, 1.02)
    ax.set_title("agreement with target's final prediction")
    mark_time_axes(ax)
    shared_legend(fig, [key("target", COLORS["target"], "--", "o"), key(name, COLORS["aligned"], "-", "o"),
                        key("control", COLORS["control"], "-", "o"), key("chance (10%)", "#b5b4af", "--")],
                  top=0.86)
    plt.show()


def fig_test_cka(name, layers):
    # X.3 — test set CKA vs target across timesteps, one panel per layer.
    fig, axes = plt.subplots(1, len(layers), figsize=(max(7.0, 4.4 * len(layers) + 0.8), 3.8), squeeze=False)
    for ax, layer in zip(axes[0], layers):
        for m, color in ((name, COLORS["aligned"]), (CONTROL, COLORS["control"])):
            x, y = test_cka(m, layer)
            ax.plot(x, y, "-", color=color, marker="o")
        ax.set_title(f"{layer}: CKA vs target (test set)")
        mark_time_axes(ax)
    axes[0][0].set_ylabel("biased CKA, full test set")
    shared_legend(fig, [key(f"{name} vs target", COLORS["aligned"], "-", "o"),
                        key("control vs target", COLORS["control"], "-", "o")], top=0.84)
    plt.show()


def fig_unaligned_layers(name):
    # X.4 — every layer, including the ones NOT in this run's loss (monitor only).
    al, ctl = hist[name], hist[CONTROL]
    in_loss = set(aligned_layers(name))
    fig, axes = plt.subplots(2, len(LAYERS), figsize=(4.4 * len(LAYERS) + 0.8, 7.2), squeeze=False)
    for j, layer in enumerate(LAYERS):
        tag = "aligned" if layer in in_loss else "NOT aligned"
        ax = axes[0][j]
        for df, color in ((al, COLORS["aligned"]), (ctl, COLORS["control"])):
            ax.plot(df["epoch"], df[f"val_cka_{KEY[layer]}"], "-", color=color)
        ax.set_title(f"{layer} ({tag}) — val batch CKA (t={LAST_T})")
        epoch_axis(ax)
        ax = axes[1][j]
        for m, color in ((name, COLORS["aligned"]), (CONTROL, COLORS["control"])):
            x, y = test_cka(m, layer)
            ax.plot(x, y, "-", color=color, marker="o")
        ax.set_title(f"{layer} ({tag}) — test CKA over time")
        mark_time_axes(ax)
    shared_legend(fig, [key(f"{name} vs target", COLORS["aligned"]),
                        key("control vs target", COLORS["control"])], top=0.92)
    plt.show()

## 0 — The target itself

### Figure 0.1 — Target accuracy against the ground truth, across timesteps (test set)

**What is plotted**: each image is shown at every one of the 20 timesteps; at each timestep the target outputs logits, and the argmax is compared with the true Imagenette label. Each point = accuracy over all test images at that timestep. Grey dashed line = chance (10%); grey shading = timesteps without CE supervision during the target's own training (reaction time); dotted line = the last timestep, whose prediction is the pseudo-label.

**Why**: this is the ceiling the students inherit. They learn the target's predictions, not the true labels, so every later number should be read relative to this curve. (The value at the last timestep should be ≈ 0.64; a large deviation would mean something in the pipeline differs from the target's training.)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x, y = test_metric("target", "gt_accuracy")
ax.plot(x, y, "--", color=COLORS["target"], marker="o")
ax.axhline(0.1, color="#b5b4af", linestyle="--", linewidth=1)
ax.set_ylim(0, 1)
ax.set_ylabel("test accuracy (ground truth)")
ax.set_title("target accuracy across timesteps")
mark_time_axes(ax)
plt.tight_layout()
plt.show()
print(f"target accuracy at t={LAST_T}: {summary['target']['gt_accuracy']:.4f} | "
      f"online target vs cached pseudo-label agreement: {summary['target']['online_vs_cache_agreement']:.4f}")

## A — `aligned_IT_w1` vs control

CE + 1.0 · log(1 − CKA_IT). The round-1 setting, re-run in the new pipeline (no data augmentation).

`RUN = "aligned_IT_w1"` below; the same figure functions are used in sections A, B and C.

In [ ]:
RUN = "aligned_IT_w1"
assert RUN in RUNS, f"{RUN} not among the evaluated runs: {list(RUNS)}"

### Figure A.1 — During training (val set, x-axis = epoch)

Three figures sharing the epoch axis, to be read together:

1. **CE vs pseudo-label** — solid = val, dashed = train (both averaged over the supervised timesteps `ce_first_timestep`…19, exactly the training loss). How well each student learns the target's labels. Train falling while val rises = overfitting (more likely now that augmentation is gone), and then the last checkpoint is not the best one.
2. **val batch CKA vs target** at the last timestep (IT). CKA on batches of 32 images, averaged — a training **monitor**, inflated by the small batch; the real CKA is in Figure A.3 (full test set).
3. **Gradient decomposition** — once per epoch, on one fixed batch of training images: the norm of the gradient of each loss term alone, *with its weight* (left), and the ratio |CKA gradients| / |CE gradient| (right, >1 = the CKA side dominates the update). Measured before clipping. The control has only a CE term (orange dashed, for reference).

**How to read them together**: when the CKA approaches 1, the gradient of log(1 − CKA) grows like weight / (1 − CKA). If the ratio climbs above 1 at the same time as the CE stops improving, the alignment term is crowding out the classification term.

In [ ]:
fig_training(RUN)

### Figure A.2 — Behavior on the test set (x-axis = timestep)

All three panels: every test image, evaluated at each of the 20 timesteps.

- **Left — accuracy vs ground truth**: target (grey dashed), this aligned student, control. Are the three systems at the same level of real recognition ability?
- **Middle — CE vs the target's pseudo-label** (log scale; values at the grey, unsupervised timesteps are naturally large). Continuous: also reflects *how confident* the student is in the target's answer.
- **Right — agreement with the target's final prediction**: fraction of images whose argmax equals the pseudo-label. Discrete: only right/wrong. The target's own line shows how its earlier timesteps agree with its final answer — the student that follows it more closely shares more of the target's temporal dynamics.

**CE vs agreement**: they usually move together but need not. A large CE gap with a small agreement gap at the same timestep means the student mostly picks the same class but with less confidence.

In [ ]:
fig_behavior(RUN)

### Figure A.3 — IT CKA vs target on the test set (x-axis = timestep)

Each point is ONE CKA computed over all 3925 test images at once (not a batch average): this student's IT representation vs the target's IT, at that timestep. Blue = this aligned student, orange = control.

**How to read it**: the alignment was applied only at the last timestep (dotted line). The gap between the lines there is the direct effect; a gap at earlier timesteps means the alignment spread to timesteps that were not constrained.

In [ ]:
fig_test_cka(RUN, ["IT"])

### Figure A.4 — Layers that were NOT aligned: did they move anyway?

This run's loss contains **only IT**. V1 and V4 were never in the loss — their CKA was only recorded (training, top row) and measured on the test set (bottom row). IT is shown for reference.

- **Top — val batch CKA vs target across epochs** (training monitor, last timestep).
- **Bottom — test-set CKA vs target across timesteps**.

**How to read it**: if the aligned student's V1/V4 lines sit above the control's, aligning IT also pulled the earlier layers toward the target (the gradient of the IT term flows through V1 and V4). If they coincide with the control, the alignment stayed local to IT.

In [ ]:
fig_unaligned_layers(RUN)

## B — `aligned_IT_w0p1` vs control

CE + 0.1 · log(1 − CKA_IT). Same as A except the CKA weight. Question: with a smaller CKA weight, does the aligned student's CE go further down (closer to the control), and how much CKA does it give up?

`RUN = "aligned_IT_w0p1"` below; the same figure functions are used in sections A, B and C.

In [ ]:
RUN = "aligned_IT_w0p1"
assert RUN in RUNS, f"{RUN} not among the evaluated runs: {list(RUNS)}"

### Figure B.1 — During training (val set, x-axis = epoch)

Three figures sharing the epoch axis, to be read together:

1. **CE vs pseudo-label** — solid = val, dashed = train (both averaged over the supervised timesteps `ce_first_timestep`…19, exactly the training loss). How well each student learns the target's labels. Train falling while val rises = overfitting (more likely now that augmentation is gone), and then the last checkpoint is not the best one.
2. **val batch CKA vs target** at the last timestep (IT). CKA on batches of 32 images, averaged — a training **monitor**, inflated by the small batch; the real CKA is in Figure B.3 (full test set).
3. **Gradient decomposition** — once per epoch, on one fixed batch of training images: the norm of the gradient of each loss term alone, *with its weight* (left), and the ratio |CKA gradients| / |CE gradient| (right, >1 = the CKA side dominates the update). Measured before clipping. The control has only a CE term (orange dashed, for reference).

**How to read them together**: when the CKA approaches 1, the gradient of log(1 − CKA) grows like weight / (1 − CKA). If the ratio climbs above 1 at the same time as the CE stops improving, the alignment term is crowding out the classification term.

In [ ]:
fig_training(RUN)

### Figure B.2 — Behavior on the test set (x-axis = timestep)

All three panels: every test image, evaluated at each of the 20 timesteps.

- **Left — accuracy vs ground truth**: target (grey dashed), this aligned student, control. Are the three systems at the same level of real recognition ability?
- **Middle — CE vs the target's pseudo-label** (log scale; values at the grey, unsupervised timesteps are naturally large). Continuous: also reflects *how confident* the student is in the target's answer.
- **Right — agreement with the target's final prediction**: fraction of images whose argmax equals the pseudo-label. Discrete: only right/wrong. The target's own line shows how its earlier timesteps agree with its final answer — the student that follows it more closely shares more of the target's temporal dynamics.

**CE vs agreement**: they usually move together but need not. A large CE gap with a small agreement gap at the same timestep means the student mostly picks the same class but with less confidence.

In [ ]:
fig_behavior(RUN)

### Figure B.3 — IT CKA vs target on the test set (x-axis = timestep)

Each point is ONE CKA computed over all 3925 test images at once (not a batch average): this student's IT representation vs the target's IT, at that timestep. Blue = this aligned student, orange = control.

**How to read it**: the alignment was applied only at the last timestep (dotted line). The gap between the lines there is the direct effect; a gap at earlier timesteps means the alignment spread to timesteps that were not constrained.

In [ ]:
fig_test_cka(RUN, ["IT"])

### Figure B.4 — Layers that were NOT aligned: did they move anyway?

This run's loss contains **only IT**. V1 and V4 were never in the loss — their CKA was only recorded (training, top row) and measured on the test set (bottom row). IT is shown for reference.

- **Top — val batch CKA vs target across epochs** (training monitor, last timestep).
- **Bottom — test-set CKA vs target across timesteps**.

**How to read it**: if the aligned student's V1/V4 lines sit above the control's, aligning IT also pulled the earlier layers toward the target (the gradient of the IT term flows through V1 and V4). If they coincide with the control, the alignment stayed local to IT.

In [ ]:
fig_unaligned_layers(RUN)

## C — `aligned_V1V4IT_w0p1` vs control

CE + 0.1 · [log(1 − CKA_V1) + log(1 − CKA_V4) + log(1 − CKA_IT)]. Same IT term as B, plus V1 and V4 terms.

`RUN = "aligned_V1V4IT_w0p1"` below; the same figure functions are used in sections A, B and C.

In [ ]:
RUN = "aligned_V1V4IT_w0p1"
assert RUN in RUNS, f"{RUN} not among the evaluated runs: {list(RUNS)}"

### Figure C.1 — During training (val set, x-axis = epoch)

Three figures sharing the epoch axis, to be read together:

1. **CE vs pseudo-label** — solid = val, dashed = train (both averaged over the supervised timesteps `ce_first_timestep`…19, exactly the training loss). How well each student learns the target's labels. Train falling while val rises = overfitting (more likely now that augmentation is gone), and then the last checkpoint is not the best one.
2. **val batch CKA vs target** at the last timestep, one panel per aligned layer (V1, V4, IT). CKA on batches of 32 images, averaged — a training **monitor**, inflated by the small batch; the real CKA is in Figure C.3 (full test set).
3. **Gradient decomposition** — once per epoch, on one fixed batch of training images: the norm of the gradient of each loss term alone, *with its weight* (left), and the ratio |CKA gradients| / |CE gradient| (right, >1 = the CKA side dominates the update). Measured before clipping. The control has only a CE term (orange dashed, for reference).

**How to read them together**: when the CKA approaches 1, the gradient of log(1 − CKA) grows like weight / (1 − CKA). If the ratio climbs above 1 at the same time as the CE stops improving, the alignment term is crowding out the classification term.

In [ ]:
fig_training(RUN)

### Figure C.2 — Behavior on the test set (x-axis = timestep)

All three panels: every test image, evaluated at each of the 20 timesteps.

- **Left — accuracy vs ground truth**: target (grey dashed), this aligned student, control. Are the three systems at the same level of real recognition ability?
- **Middle — CE vs the target's pseudo-label** (log scale; values at the grey, unsupervised timesteps are naturally large). Continuous: also reflects *how confident* the student is in the target's answer.
- **Right — agreement with the target's final prediction**: fraction of images whose argmax equals the pseudo-label. Discrete: only right/wrong. The target's own line shows how its earlier timesteps agree with its final answer — the student that follows it more closely shares more of the target's temporal dynamics.

**CE vs agreement**: they usually move together but need not. A large CE gap with a small agreement gap at the same timestep means the student mostly picks the same class but with less confidence.

In [ ]:
fig_behavior(RUN)

### Figure C.3 — V1, V4 and IT CKA vs target on the test set (x-axis = timestep)

Each point is ONE CKA computed over all 3925 test images at once (not a batch average): this student's layer representation vs the target's same layer, at that timestep. Blue = this aligned student, orange = control.

**How to read it**: the alignment was applied only at the last timestep (dotted line). The gap between the lines there is the direct effect; a gap at earlier timesteps means the alignment spread to timesteps that were not constrained. Compare the three panels to see which layers are easy to align and how far the control already is from the target in each.

In [ ]:
fig_test_cka(RUN, LAYERS)

## D — Comparison across configurations

All four runs together. Colors: one per run (control orange; aligned runs in blues/purple), the same in every figure of this section.

### D.1 — Summary table (last checkpoint)

- **val CE (last epoch)** / **train CE (last epoch)**: from training (val set). Gap train < val = overfitting.
- **test agreement / GT accuracy / CE**: test set, last timestep.
- **CKA vs target (V1, V4, IT)**: biased CKA, full test set, last timestep. Which of these layers were in the loss is in the `loss` column.

In [ ]:
rows = []
for n, r in RUNS.items():
    s = summary["students"][n]
    h = hist[n].iloc[-1]
    row = {
        "run": n,
        "loss": "CE only" if r["is_control"] else
                "CE + " + " + ".join(f"{w:g}·{k.split('_')[0]}" for k, w in r["loss_weights"].items()),
        "val CE (last epoch)": h["val_ce_loss"],
        "train CE (last epoch)": h["train_ce_loss"],
        f"test agreement (t={LAST_T})": s["agreement"],
        f"test GT accuracy (t={LAST_T})": s["gt_accuracy"],
        f"test CE (t={LAST_T})": s["ce_pseudo"],
    }
    for layer in LAYERS:
        row[f"CKA {layer}"] = s["cka_biased_vs_target"][layer]
    rows.append(row)
table = pd.DataFrame(rows).set_index("run")
print(f"target GT accuracy (t={LAST_T}): {summary['target']['gt_accuracy']:.3f}")
table.style.format({c: "{:.3f}" for c in table.columns if c != "loss"})

### D.2 — val CE across epochs, all runs

The question behind configuration B: does lowering the CKA weight bring the aligned student's CE closer to the control's? Solid = val, dashed = train.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for n in RUNS:
    df = hist[n]
    ax.plot(df["epoch"], df["val_ce_loss"], "-", color=RUN_COLORS[n])
    ax.plot(df["epoch"], df["train_ce_loss"], "--", color=RUN_COLORS[n], alpha=0.6)
ax.set_title("CE vs pseudo-label")
epoch_axis(ax)
shared_legend(fig, [key(n, RUN_COLORS[n]) for n in RUNS] +
              [key("val", "#52514e", "-"), key("train", "#52514e", "--")], top=0.84)
plt.show()

### D.3 — IT CKA vs target on the test set across timesteps, all runs

Same quantity as Figures A.3 / B.3 / C.3 (IT panel), all runs on one axis: how much IT alignment each configuration buys, and whether adding the V1 and V4 terms (C) changes IT compared with aligning IT alone at the same weight (B).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for n in RUNS:
    x, y = test_cka(n, "IT")
    ax.plot(x, y, "-", color=RUN_COLORS[n], marker="o")
ax.set_ylabel("biased CKA vs target, full test set")
ax.set_title("IT: CKA vs target across timesteps")
mark_time_axes(ax)
shared_legend(fig, [key(n, RUN_COLORS[n], "-", "o") for n in RUNS], ncol=2, top=0.82)
plt.show()

### D.4 — Gradients across configurations

- **Left**: |CKA gradients| / |CE gradient| on the fixed diagnostic batch, per aligned run (log scale; above the dashed line = the CKA side dominates).
- **Middle**: mean total gradient norm per training step, *before* clipping, all runs. Dashed line = the clipping threshold (`max_grad_norm`).
- **Right**: fraction of steps that were clipped.

**Why the middle and right panels matter**: clipping rescales the whole gradient (CE and CKA alike, so their ratio is unchanged), but the weight decay is added *after* clipping. A run whose gradients are clipped harder therefore gets relatively stronger weight decay. Large differences between runs here are a side effect of the CKA weight to keep in mind when comparing them.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for n in ALIGNED:
    axes[0].plot(hist[n]["epoch"], hist[n]["diag_ratio_cka_to_ce"], "-", color=RUN_COLORS[n])
axes[0].axhline(1.0, color="#b5b4af", linestyle="--", linewidth=1)
axes[0].set_yscale("log")
axes[0].set_title("|CKA gradients| / |CE gradient|")
for n in RUNS:
    axes[1].plot(hist[n]["epoch"], hist[n]["train_grad_norm"], "-", color=RUN_COLORS[n])
    axes[2].plot(hist[n]["epoch"], hist[n]["train_clip_fraction"], "-", color=RUN_COLORS[n])
import yaml
max_norm = yaml.safe_load((Path(RUNS[CONTROL]["dir"]) / "resolved_config.yaml").read_text())["optimization"]["max_grad_norm"]
if max_norm is not None:
    axes[1].axhline(max_norm, color="#b5b4af", linestyle="--", linewidth=1)
axes[1].set_yscale("log")
axes[1].set_title("mean gradient norm per step (before clipping)")
axes[2].set_ylim(-0.02, 1.02)
axes[2].set_title("fraction of steps clipped")
for ax in axes:
    epoch_axis(ax)
shared_legend(fig, [key(n, RUN_COLORS[n]) for n in RUNS], top=0.84)
plt.show()